# Dia 03 — Ferramentas de linha de comando

**🃏 Cartas:** 3 · 24 · 45 &nbsp;·&nbsp; **📘 Original:** *Day 3. Install and run Python command line tools*
&nbsp;·&nbsp; [repositório](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias) &nbsp;·&nbsp; [curso original do cipher387](https://github.com/cipher387/python-for-OSINT-21-days)

---

## 📖 Capítulo 3 — Ferramenta de campo

Quinta-feira, 10h20. Em cinco depoimentos diferentes aparece a mesma assinatura no WhatsApp:
*"Kau"*. E num print de Instagram que uma vítima mandou, um apelido: **`kau.ofertas`**.

— Apelido é ouro — diz a Iara. — Gente que aplica golpe troca de chip, de chave PIX, de site. Mas o
apelido, por preguiça ou por vaidade, costuma ficar.

Ela abre o terminal dela e digita um comando comprido. Uma lista de sites começa a rolar na tela.

— Isso é o blackbird. Alguém já escreveu uma ferramenta que pergunta a centenas de sites se um
nome de usuário existe. A gente não precisa reinventar a roda. Mas precisa **saber rodar a roda**:
instalar, passar as opções certas, chamar de dentro de um script e, principalmente, **registrar o que
foi feito**. Se um dia isso virar prova, alguém vai perguntar: *que comando você rodou, quando e o que
saiu?*

Você anota a pergunta num post-it e cola no monitor.

— E outra coisa — ela completa, baixando a tela. — O apelido do caso pode ser de um inocente em algum
site. Primeiro a gente aprende com um alvo de treino. Depois, com autorização, no alvo de verdade.

In [ ]:
#@title ⚙️ Preparar o ambiente (rode esta célula primeiro)
# No Colab: baixa o repositório e entra na pasta do dia.
# No Jupyter local: o notebook já está na pasta do dia, então não faz nada.
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias"
if "google.colab" in sys.modules:
    destino = "/content/python-para-osint-21-dias"
    if not os.path.isdir(destino):
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    os.chdir(f"{destino}/dia-03-ferramentas-de-linha-de-comando")

PASTA_NB = str(Path.cwd())          # usada pelas células para voltar sempre à pasta certa
print("📂 Pasta do dia:", PASTA_NB)
print("🐍 Python", sys.version.split()[0])

## 🎯 Objetivos

- Instalar uma ferramenta de OSINT pelo **PyPI** (`pip install`) e pelo **GitHub** (`git clone` +
  `requirements.txt`).
- Rodar o **blackbird** pelo terminal e de dentro de um script com `subprocess`.
- Criar a sua própria ferramenta com opções (`--cep`, `--formato`) usando `argparse`.
- Rodar programas com `subprocess.run`, capturar a saída e **registrar** cada execução.
- Entender como uma ferramenta decide se um username "existe" num site.

## 🧠 Conceito em 2 minutos

Uma **ferramenta de linha de comando** (CLI) é um programa que você usa digitando, em vez de clicar.
Tudo o que vem depois do nome do programa são **argumentos**:

```
python blackbird.py   --username   ivanov
└──────┬──────────┘   └───┬─────┘  └──┬──┘
    programa            opção        valor
```

Há três jeitos de colocar uma ferramenta Python para rodar:

| Jeito | Comando | Quando |
|---|---|---|
| **PyPI** | `pip install nome` e depois `nome alvo` | o autor publicou no PyPI |
| **GitHub** | `git clone ...`, `pip install -r requirements.txt`, `python ferramenta.py` | só existe o repositório |
| **Do seu script** | `subprocess.run([...])` | para automatizar e registrar |

`requirements.txt` é a lista de bibliotecas que a ferramenta precisa. E o **código de retorno**
(`returncode`) é como um programa avisa se deu certo: `0` é sucesso; qualquer outro número é erro.

**Como essas ferramentas "acham" um perfil?** Elas montam a URL do perfil (`https://site/{usuario}`) e
olham a resposta: status **200** costuma ser "existe", **404** "não existe". Mas muitos sites mentem
(respondem 200 para tudo), então as boas ferramentas também procuram um **texto** na página. O projeto
comunitário **WhatsMyName** guarda essas regras para mais de 700 sites, e o blackbird usa essa base.

## 🥋 Nível 1 — Exercício original (cipher387)

| Arquivo | O que faz |
|---|---|
| [`original/launch.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-03-ferramentas-de-linha-de-comando/original/launch.py) | Roda `python blackbird.py -u ivanov` de dentro do Python com `subprocess.call` |

### Instalar e rodar o blackbird (checado em outubro de 2026)

O repositório continua em <https://github.com/p1ngul1n0/blackbird>. A opção `-u` / `--username` do
curso original **continua igual**, então não foi preciso criar uma versão 2026 do `launch.py`.

```bash
cd dia-03-ferramentas-de-linha-de-comando
git clone https://github.com/p1ngul1n0/blackbird
cd blackbird
pip install -r requirements.txt
python blackbird.py --username cipher387          # o mesmo que -u
python blackbird.py --username cipher387 --csv    # também exporta CSV (há --pdf e --json)
```

Para usar o `launch.py` como o autor ensina: **copie** o arquivo para dentro da pasta `blackbird/` e rode
de lá (no macOS/Linux, se `python` não existir, edite o texto do comando para `python3`):

```bash
cp ../original/launch.py .
python launch.py
```

No **Colab**, as mesmas linhas funcionam numa célula com `!` na frente
(`!git clone ...`, `%cd blackbird`, `!pip install -r requirements.txt`, `!python blackbird.py -u cipher387`).
Atenção: a pasta `blackbird/` **não** está no `.gitignore`. Se você estiver num fork, não faça commit dela.

> ℹ️ **O que mudou desde o PDF.** O blackbird hoje usa a base **WhatsMyName**, exporta CSV/PDF/JSON e
> tem um recurso opcional de IA (`--ai`) que manda para um serviço externo a lista de sites onde o
> username foi achado. Num caso real, pense duas vezes antes de usar: é dado do caso saindo da sua
> máquina. O **Thorndyke**, a outra ferramenta citada no PDF, foi **arquivado** pelo autor em 2022 e não
> aparece mais no PyPI; pule essa parte. O próprio cipher387 lembra que **Sherlock** e **Maigret**
> cobrem mais sites (`pip install sherlock-project`, `pip install maigret`).

### `original/launch.py` — Rodar o blackbird de dentro do Python com subprocess

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Rodar o blackbird de dentro do Python com subprocess
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_3/launch.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# Requer: o blackbird clonado (git clone https://github.com/p1ngul1n0/blackbird),
#         pip install -r requirements.txt, e este arquivo DENTRO da pasta blackbird/.
# ──────────────────────────────────────────────────────────────────────────

# 💬 subprocess vem com o Python: serve para rodar outros programas como se você
# 💬 tivesse digitado o comando no terminal.
import subprocess
# 💬 Roda "python blackbird.py -u ivanov": procura o nome de usuário "ivanov" em
# 💬 centenas de sites. shell=True entrega o texto inteiro para o terminal interpretar.
# 💬 Em 2026 a opção -u (ou --username) continua igual no blackbird.
# 💬 No macOS/Linux, se "python" não existir, troque por "python3" no texto do comando.
subprocess.call("python blackbird.py -u ivanov", shell=True)

## 🇧🇷 Nível 2 — Camada Brasil

| Arquivo | O que faz |
|---|---|
| [`br/01_cli_cep.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-03-ferramentas-de-linha-de-comando/br/01_cli_cep.py) | Sua primeira CLI: `--cep` e `--formato texto/json/csv`, com `--help` automático |
| [`br/02_rodar_e_registrar.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-03-ferramentas-de-linha-de-comando/br/02_rodar_e_registrar.py) | Roda o `01` com `subprocess.run` e grava um log forense em `saidas/log_execucoes.jsonl` |
| [`br/03_mini_blackbird.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-03-ferramentas-de-linha-de-comando/br/03_mini_blackbird.py) | **O caso:** um blackbird de bolso que verifica um username no GitHub, GitLab e Codeberg |

```bash
cd dia-03-ferramentas-de-linha-de-comando
python br/01_cli_cep.py --cep 30130-010
python br/01_cli_cep.py --cep 30130010 --formato json
python br/01_cli_cep.py --help
python br/02_rodar_e_registrar.py
python br/03_mini_blackbird.py --username cipher387
OSINT_OFFLINE=1 python br/03_mini_blackbird.py       # usa a amostra de dados/
```

**O que você deve ver:** o `01` mostra o endereço do CEP 30130-010, no Centro de BH (o mesmo CEP que
aparece no rodapé da loja; no Dia 5 você investiga esse endereço a fundo). O `02` roda três CEPs (um
deles inválido de propósito, para você ver o código de retorno `2`) e o log ganha três linhas a cada
execução, cada uma com data/hora, comando e o SHA-256 da saída. O `03` acha o `cipher387` no GitHub.

> 💡 Repare que o `01` manda os avisos para o **stderr** e o resultado para o **stdout**. É por isso que o
> `02` consegue guardar o JSON limpo, separado do aviso `⚠️ MODO OFFLINE`.

> 💡 Os campos com formulário à direita (`#@param`) podem ser editados sem mexer no código. Para testar sem internet, rode antes: `os.environ['OSINT_OFFLINE'] = '1'`.

### `br/01_cli_cep.py` — Sua primeira ferramenta de linha de comando: consultar um CEP com --cep e --formato.

In [ ]:
%cd -q {PASTA_NB}
"""
Sua primeira ferramenta de linha de comando: consultar um CEP com --cep e --formato.

Carta 24 (rank B) — Jutsu da Ferramenta de Campo (parte 1)

O que você aprende:
  • o que são "argumentos de linha de comando" (o que vem depois do nome do script);
  • a criar opções como --cep e --formato com argparse (nativo), com ajuda automática (--help);
  • a validar a entrada antes de gastar uma consulta;
  • a devolver o resultado em formatos diferentes: texto para gente, JSON ou CSV para máquina.

Usa a BrasilAPI (https://brasilapi.com.br/api/cep/v2/{cep}). No Dia 5 você aprende HTTP a fundo;
aqui o foco é a "casca" da ferramenta.

Como rodar (dentro da pasta dia-03-ferramentas-de-linha-de-comando):
    python br/01_cli_cep.py                         # usa o CEP padrão (parâmetro abaixo)
    python br/01_cli_cep.py --cep 30130-010
    python br/01_cli_cep.py --cep 30130010 --formato json
    python br/01_cli_cep.py --help                  # a ajuda que o argparse escreve sozinho
    OSINT_OFFLINE=1 python br/01_cli_cep.py         # sem internet, usa dados/
"""

import argparse             # o "leitor" de argumentos de linha de comando
import json
import os
import sys                  # sys.exit para sair com código de erro
import urllib.request
from pathlib import Path

# ── Parâmetros (valem quando você NÃO passa --cep / --formato) ────────────
CEP_PADRAO = "30130-010"  #@param {type:"string"}
FORMATO_PADRAO = "texto"  #@param ["texto", "json", "csv"]

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
OFFLINE = os.environ.get("OSINT_OFFLINE") == "1"


def buscar_json(url, amostra):
    """Baixa um JSON da internet; se falhar (ou OFFLINE=1), lê a amostra local."""
    if not OFFLINE:
        try:
            pedido = urllib.request.Request(url, headers={"User-Agent": "python-para-osint-21-dias/estudo"})
            with urllib.request.urlopen(pedido, timeout=15) as resposta:
                return json.load(resposta)
        except Exception as erro:
            # Mensagens de aviso vão para stderr: assim não "sujam" a saída JSON/CSV (stdout).
            print(f"⚠️ MODO OFFLINE: a consulta falhou ({erro}). Usando a amostra local.", file=sys.stderr)
    else:
        print("⚠️ MODO OFFLINE: usando a amostra local.", file=sys.stderr)
    return json.loads((PASTA_DIA / "dados" / amostra).read_text(encoding="utf-8"))


# ── 1. Descrever as opções da ferramenta ──────────────────────────────────
# O ArgumentParser monta, sozinho, a mensagem de --help e as mensagens de erro.
parser = argparse.ArgumentParser(
    prog="cli_cep",
    description="Consulta um CEP na BrasilAPI e mostra o endereço.",
    epilog="Exemplo: python br/01_cli_cep.py --cep 30130-010 --formato json",
)
parser.add_argument("--cep", default=CEP_PADRAO,
                    help=f"CEP com ou sem traço (padrão: {CEP_PADRAO})")
parser.add_argument("--formato", choices=["texto", "json", "csv"], default=FORMATO_PADRAO,
                    help="como mostrar o resultado (padrão: %(default)s)")

# parse_known_args (em vez de parse_args) ignora argumentos "estranhos". Isso evita erro
# no Colab/Jupyter, que passa os próprios argumentos escondidos para o Python.
args, _ = parser.parse_known_args()

# ── 2. Validar ANTES de consultar ─────────────────────────────────────────
# Ficamos só com os dígitos; um CEP tem exatamente 8.
cep = "".join(c for c in args.cep if c.isdigit())
if len(cep) != 8:
    print(f"❌ CEP inválido: {args.cep!r} (precisa de 8 dígitos).", file=sys.stderr)
    sys.exit(2)                 # código 2 = "uso errado" (convenção de ferramentas de terminal)

# ── 3. Consultar ──────────────────────────────────────────────────────────
# Só existe amostra offline do CEP 30130-010. Para outro CEP sem internet, avisamos.
amostra = "cep_30130010.json"
if (OFFLINE and cep != "30130010"):
    print(f"⚠️  Não há amostra para {cep}; mostrando a do 30130-010 só como exemplo.", file=sys.stderr)
dados = buscar_json(f"https://brasilapi.com.br/api/cep/v2/{cep}", amostra)

# Os campos que nos interessam (usamos .get para não quebrar se algum faltar).
campos = {
    "cep": dados.get("cep", cep),
    "logradouro": dados.get("street", ""),
    "bairro": dados.get("neighborhood", ""),
    "cidade": dados.get("city", ""),
    "uf": dados.get("state", ""),
    "fonte": dados.get("service", ""),
}

# ── 4. Mostrar no formato pedido ──────────────────────────────────────────
if args.formato == "json":
    # JSON: bom para outro programa ler. ensure_ascii=False mantém os acentos.
    print(json.dumps(campos, ensure_ascii=False, indent=2))
elif args.formato == "csv":
    # CSV com ";" (padrão do Excel em português): cabeçalho + uma linha.
    print(";".join(campos.keys()))
    print(";".join(str(v) for v in campos.values()))
else:
    print(f"📮 CEP {campos['cep'][:5]}-{campos['cep'][5:]}")
    print(f"   🛣️  {campos['logradouro'] or '(sem logradouro: CEP geral da cidade?)'}")
    print(f"   🏘️  {campos['bairro']} — {campos['cidade']}/{campos['uf']}")
    print(f"   🔗 fonte dos dados: {campos['fonte']}")

### `br/02_rodar_e_registrar.py` — Rodar uma ferramenta pelo Python e registrar tudo: o "diário de bordo" forense.

In [ ]:
%cd -q {PASTA_NB}
"""
Rodar uma ferramenta pelo Python e registrar tudo: o "diário de bordo" forense.

Carta 24 (rank B) — Jutsu da Ferramenta de Campo (parte 2)

O que você aprende:
  • a rodar outro programa com subprocess.run (o "irmão moderno" do subprocess.call do original);
  • por que passar o comando como LISTA, sem shell=True, é mais seguro;
  • a usar sys.executable para chamar exatamente o mesmo Python que está rodando este script;
  • a capturar a saída (stdout), os erros (stderr) e o código de retorno;
  • a gravar um log com data/hora, comando, duração e resultado: boa prática forense.

Como rodar (dentro da pasta dia-03-ferramentas-de-linha-de-comando):
    python br/02_rodar_e_registrar.py
    OSINT_OFFLINE=1 python br/02_rodar_e_registrar.py
"""

import hashlib              # para calcular a "impressão digital" (SHA-256) da saída
import json
import os
import subprocess           # rodar outros programas
import sys
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path

# ── Parâmetros ────────────────────────────────────────────────────────────
CEPS = "30130-010, 30130010, 123"  #@param {type:"string"}
FORMATO = "json"  #@param ["texto", "json", "csv"]

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
FERRAMENTA = PASTA_DIA / "br" / "01_cli_cep.py"
SAIDAS = PASTA_DIA / "saidas"
SAIDAS.mkdir(exist_ok=True)
LOG = SAIDAS / "log_execucoes.jsonl"     # .jsonl = um JSON por linha (fácil de ir acrescentando)

# Fuso de Brasília, com plano B para Windows sem tzdata.
try:
    from zoneinfo import ZoneInfo
    FUSO = ZoneInfo("America/Sao_Paulo")
except Exception:
    FUSO = timezone(timedelta(hours=-3))

print(f"🧰 Ferramenta: {FERRAMENTA.name}")
print(f"🐍 Python usado: {sys.executable}")
print(f"📝 Log: {LOG.relative_to(PASTA_DIA)}\n")

for cep in [c.strip() for c in CEPS.split(",") if c.strip()]:
    # ── 1. Montar o comando como LISTA ────────────────────────────────────
    # Cada pedaço é um item. Sem shell=True, o texto do CEP nunca é "interpretado"
    # pelo terminal: se alguém passasse "30130010; rm -rf /", seria só um CEP inválido.
    comando = [sys.executable, str(FERRAMENTA), "--cep", cep, "--formato", FORMATO]

    # ── 2. Rodar e capturar ───────────────────────────────────────────────
    inicio = datetime.now(FUSO)
    cronometro = time.perf_counter()
    resultado = subprocess.run(
        comando,
        capture_output=True,     # guarda stdout e stderr em vez de jogar na tela
        text=True,               # devolve texto (str) em vez de bytes
        encoding="utf-8",        # os acentos e emojis chegam inteiros (importante no Windows)
        env=dict(os.environ, PYTHONIOENCODING="utf-8"),  # o processo filho também fala UTF-8
        timeout=60,              # se travar, desiste depois de 60 segundos
        cwd=PASTA_DIA,           # roda "de dentro" da pasta do dia
    )
    duracao = time.perf_counter() - cronometro

    # ── 3. Mostrar ────────────────────────────────────────────────────────
    icone = "✅" if resultado.returncode == 0 else "❌"
    print(f"{icone} CEP {cep!r} → código de retorno {resultado.returncode} ({duracao:.2f} s)")
    for linha in resultado.stdout.strip().splitlines():
        print(f"   │ {linha}")
    for linha in resultado.stderr.strip().splitlines():
        print(f"   ⚠ {linha}")

    # ── 4. Registrar (uma linha JSON por execução) ────────────────────────
    # O SHA-256 da saída permite provar depois que o resultado guardado não foi alterado.
    registro = {
        "quando": inicio.isoformat(timespec="seconds"),
        "comando": comando[1:],                       # sem o caminho do Python, para ficar legível
        "python": sys.version.split()[0],
        "codigo_retorno": resultado.returncode,
        "duracao_s": round(duracao, 3),
        "modo_offline": "MODO OFFLINE" in resultado.stderr,
        "sha256_saida": hashlib.sha256(resultado.stdout.encode("utf-8")).hexdigest(),
        "saida": resultado.stdout,
        "erros": resultado.stderr,
    }
    # Modo "a" (append): acrescenta no fim, sem apagar as execuções anteriores.
    with open(LOG, "a", encoding="utf-8") as arquivo:
        arquivo.write(json.dumps(registro, ensure_ascii=False) + "\n")
    print()

# ── 5. Reler o log ────────────────────────────────────────────────────────
linhas = LOG.read_text(encoding="utf-8").splitlines()
print(f"📚 O log tem {len(linhas)} execução(ões) registradas. A última:")
ultima = json.loads(linhas[-1])
print(f"   {ultima['quando']} · {' '.join(ultima['comando'][-4:])} · retorno {ultima['codigo_retorno']}"
      f" · sha256 {ultima['sha256_saida'][:16]}…")

### `br/03_mini_blackbird.py` — Mini-blackbird: o username existe nestes sites? Uma versão de bolso, só com stdlib.

In [ ]:
%cd -q {PASTA_NB}
"""
Mini-blackbird: o username existe nestes sites? Uma versão de bolso, só com stdlib.

Carta 24 (rank B) — Jutsu da Ferramenta de Campo (parte 3)

O que você aprende:
  • como ferramentas como blackbird, Sherlock e Maigret decidem se um perfil "existe";
  • a usar o código de status HTTP (200 = achou, 404 = não achou) e, quando ele mente,
    a olhar o CONTEÚDO da resposta;
  • a descrever sites como DADOS (uma lista de dicionários), no mesmo espírito do WhatsMyName;
  • a fazer poucos pedidos, com pausa e User-Agent honesto.

As regras dos três sites abaixo seguem o projeto WhatsMyName (CC BY-SA 4.0), que é a base
do blackbird hoje.

Como rodar (dentro da pasta dia-03-ferramentas-de-linha-de-comando):
    python br/03_mini_blackbird.py
    python br/03_mini_blackbird.py --username outro_nome   # igual ao "-u" do blackbird
    OSINT_OFFLINE=1 python br/03_mini_blackbird.py          # usa a amostra de dados/
"""

import argparse
import json
import os
import time
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

# ── Parâmetros ────────────────────────────────────────────────────────────
# Padrão: o próprio autor do curso, que tem perfil público no GitHub.
USERNAME = "cipher387"  #@param {type:"string"}
PAUSA_SEGUNDOS = 1  #@param {type:"number"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
OFFLINE = os.environ.get("OSINT_OFFLINE") == "1"

# Como no script 01: a opção --username, se for passada, vence o parâmetro do topo.
parser = argparse.ArgumentParser(description="Verifica se um username existe em GitHub, GitLab e Codeberg.")
parser.add_argument("-u", "--username", default=USERNAME, help="username a procurar (padrão: %(default)s)")
USERNAME = parser.parse_known_args()[0].username

# ── 1. Os sites, descritos como dados ─────────────────────────────────────
# Para cada site: onde perguntar ("{u}" vira o username), qual status indica "existe",
# e um trecho de texto que confirma. Usamos as APIs públicas, que respondem JSON.
SITES = [
    {"nome": "GitHub", "url": "https://api.github.com/users/{u}", "perfil": "https://github.com/{u}",
     "achou_status": 200, "achou_texto": '"login"'},
    # O GitLab responde 200 SEMPRE; a diferença está no corpo: [] (lista vazia) = não existe.
    {"nome": "GitLab", "url": "https://gitlab.com/api/v4/users?username={u}", "perfil": "https://gitlab.com/{u}",
     "achou_status": 200, "achou_texto": '"id"'},
    {"nome": "Codeberg", "url": "https://codeberg.org/api/v1/users/{u}", "perfil": "https://codeberg.org/{u}",
     "achou_status": 200, "achou_texto": '"id"'},
]


def perguntar(url):
    """Devolve (status, corpo_em_texto). 404 também é uma resposta válida: o servidor RESPONDEU."""
    pedido = urllib.request.Request(url, headers={
        "User-Agent": "python-para-osint-21-dias/estudo",
        "Accept": "application/json",
    })
    try:
        with urllib.request.urlopen(pedido, timeout=15) as resposta:
            return resposta.status, resposta.read().decode("utf-8", errors="replace")
    except urllib.error.HTTPError as erro:          # 404, 403, 429... chegam aqui
        return erro.code, erro.read().decode("utf-8", errors="replace")


# ── 2. Amostra offline (respostas guardadas no formato real de cada API) ──
amostra = None
if OFFLINE:
    print("⚠️ MODO OFFLINE: usando a amostra local.")
    amostra = json.loads((PASTA_DIA / "dados" / "mini_blackbird_cipher387.json").read_text(encoding="utf-8"))
    if USERNAME != "cipher387":
        print(f"   (a amostra só existe para 'cipher387'; {USERNAME!r} não foi consultado)")
    USERNAME = "cipher387"          # a amostra só existe para este username

# Validar o username: só letras, números, ponto, hífen e sublinhado.
# Isso também evita montar URLs estranhas com espaço, "/" ou "?".
permitidos = set("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789._-")
if not USERNAME or not set(USERNAME) <= permitidos:
    raise SystemExit(f"❌ Username inválido: {USERNAME!r}")

print(f"🔎 Procurando o username {USERNAME!r} em {len(SITES)} sites\n")
print(f"{'Site':<10} {'Status':>6}  Resultado")
print("─" * 60)

# ── 3. Perguntar a cada site ──────────────────────────────────────────────
encontrados = []
for site in SITES:
    url = site["url"].format(u=urllib.parse.quote(USERNAME))
    if amostra is not None:
        status = amostra[site["nome"]]["status"]
        corpo = json.dumps(amostra[site["nome"]]["corpo"])
    else:
        try:
            status, corpo = perguntar(url)
        except Exception as erro:                   # sem internet, DNS, timeout...
            print(f"⚠️ MODO OFFLINE: {site['nome']} não respondeu ({type(erro).__name__}).")
            continue
        time.sleep(PAUSA_SEGUNDOS)                  # educação: uma pausa entre sites

    # A decisão: status certo E o texto que confirma.
    if status == site["achou_status"] and site["achou_texto"] in corpo:
        resultado = "✅ existe  → " + site["perfil"].format(u=USERNAME)
        encontrados.append(site["nome"])
    elif status in (403, 429):
        resultado = "🟡 bloqueado/limite de pedidos: tente mais tarde"
    else:
        resultado = "❌ não encontrado"
    print(f"{site['nome']:<10} {status:>6}  {resultado}")

# ── 4. Resumo e cautela ───────────────────────────────────────────────────
print("─" * 60)
print(f"📊 Encontrado em {len(encontrados)} de {len(SITES)} sites: {', '.join(encontrados) or 'nenhum'}")
print("⚖️  Mesmo username em dois sites NÃO prova que é a mesma pessoa. Confirme com outros elementos")
print("   (foto, bio, links cruzados, datas). E lembre: o servidor registra o seu IP a cada consulta.")

## ⚖️ Ética, LGPD e OPSEC

- **Ferramenta de enumeração bate na porta de centenas de sites com o seu IP.** Cada site registra o
  acesso. No trabalho, use a infraestrutura autorizada; no estudo, use alvos de treino públicos
  (como `cipher387`, o autor do curso).
- **Apelido de suspeito pode ser de um inocente.** "kau.ofertas" é fictício, mas pode existir como conta
  real de outra pessoa. Por isso o desafio só **simula** a busca dele. Consultar alguém sem finalidade
  definida é tratamento de dado pessoal sem base legal (LGPD, art. 7º).
- **Achar o mesmo username não prova identidade.** Trate como hipótese e procure elementos que
  confirmem (foto, bio, links cruzados, datas).
- **Registre o que você rodou.** Comando, versão da ferramenta, data/hora com fuso e o hash da saída. Isso
  é o começo de uma cadeia de custódia (você aprofunda no Dia 14).
- **Leia o código antes de rodar.** `git clone` + `pip install -r requirements.txt` executa código de
  terceiros na sua máquina. Prefira repositórios conhecidos e um ambiente separado (Colab, Codespaces,
  `venv` ou máquina virtual).
- **Recursos de IA embutidos enviam dados para fora.** Antes de usar `--ai` ou similares, confira o que
  é enviado e se isso é permitido no seu caso.

## 🎯 Quiz rápido

Escolha a resposta no formulário de cada célula e rode a célula.

In [ ]:
#@title 1. Por que o br/02 passa o comando como lista ([sys.executable, 'br/01_cli_cep.py', '--cep', cep]) e sem shell=True?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Porque listas são mais rápidas", "B) Para que o texto digitado nunca seja interpretado pelo terminal como outro comando", "C) Porque subprocess.run não aceita texto", "D) Para esconder o comando do log"]
correta = "B) Para que o texto digitado nunca seja interpretado pelo terminal como outro comando"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "Com shell=True, um valor como '30130010; rm -rf ~' viraria dois comandos. Em lista, cada item é só um argumento.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 2. No mini-blackbird, por que o GitLab precisa da checagem de texto além do status HTTP?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Porque o GitLab sempre responde 404", "B) Porque a API responde 200 mesmo quando o usuário não existe, devolvendo uma lista vazia []", "C) Porque o GitLab exige senha", "D) Porque o status HTTP do GitLab vem em texto"]
correta = "B) Porque a API responde 200 mesmo quando o usuário não existe, devolvendo uma lista vazia []"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "A busca /api/v4/users?username= devolve 200 com [] quando não acha. Por isso a regra exige também o texto '\"id\"'.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 3. O mesmo username apareceu no GitHub e no Codeberg. O que você pode concluir?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Que é a mesma pessoa", "B) Que a pessoa é desenvolvedora", "C) Nada definitivo: é uma hipótese a confirmar com outros elementos", "D) Que o perfil é falso"]
correta = "C) Nada definitivo: é uma hipótese a confirmar com outros elementos"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "Usernames se repetem por coincidência. Confirme com foto, bio, links cruzados e datas antes de afirmar qualquer vínculo.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

# 🏆 Desafio S — Dia 3: Lote de buscas com registro

**Carta 45 · Rank S**

## A missão

O vendedor assina "Kau" e usa o apelido **`kau.ofertas`**. Golpista troca de conta, mas costuma repetir o
apelido com pequenas variações: `kau_ofertas`, `kau-ofertas`, `kauofertas`...

A Iara quer um **lote**: um script que gera as variantes, monta o comando da ferramenta de campo para
cada uma e **registra tudo**. Mas ela impõe uma regra:

> "O apelido do caso é inventado, só que pode existir de verdade, como conta de um desconhecido. Então
> para ele a gente **só simula**. Para provar que o lote funciona, roda de verdade num alvo de treino
> público: `cipher387`."

Escreva um script que:

1. Gere as variantes de `APELIDO_DO_CASO` trocando o separador por `.`, `_`, `-` e nenhum.
2. Para cada variante, monte o comando `br/03_mini_blackbird.py --username <variante>` e **só registre**
   (modo simulação).
3. Para `ALVO_DE_TREINO`, rode o comando de verdade com `subprocess.run` (lista, **sem** `shell=True`)
   e capture o resultado de cada site.
4. Grave `saidas/lote_buscas.csv` (separador `;`, `utf-8-sig`) com data/hora de Brasília, username,
   modo, código de retorno e resultado.
5. Acrescente (modo `"a"`) uma linha por comando em `saidas/log_lote.txt`.

## Critérios de rank S

- [ ] Só biblioteca padrão
- [ ] Comando como lista e `sys.executable` (nada de `"python ..."` com `shell=True`)
- [ ] Nenhuma consulta real ao apelido do caso
- [ ] O log cresce a cada execução, sem apagar o anterior
- [ ] Roda com `OSINT_OFFLINE=1`

## Dica

Um `set` elimina variantes repetidas: `{sep.join(partes) for sep in (".", "_", "-", "")}`.
E o processo filho herda as variáveis de ambiente: se você rodar com `OSINT_OFFLINE=1`, o
mini-blackbird também entra no modo offline sozinho.

In [ ]:
%cd -q {PASTA_NB}
# ✍️ Escreva sua solução aqui.

In [ ]:
#@title 🔒 Gabarito (dê dois cliques aqui para ver o código) { display-mode: "form" }
%cd -q {PASTA_NB}
"""
Gabarito do Desafio S do Dia 3 — Lote de buscas com registro.

Golpistas trocam de conta, mas costumam repetir o apelido com pequenas variações.
Este script:
  1. gera variantes de um apelido (separadores . _ - e sem separador);
  2. monta, para cada variante, o comando do br/03_mini_blackbird.py (como LISTA, sem shell=True);
  3. em modo SIMULAÇÃO (padrão para o apelido fictício do caso) só registra os comandos;
     para o alvo de treino público (cipher387), roda de verdade e captura a saída;
  4. grava saidas/lote_buscas.csv (abre no Excel) e acrescenta cada execução em saidas/log_lote.txt.

Por que simular o apelido do caso? "kau.ofertas" é inventado, mas pode existir de verdade,
como conta de um desconhecido, em algum site. Consultá-lo seria investigar um inocente.
"""

import csv
import os
import subprocess
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path

APELIDO_DO_CASO = "kau.ofertas"  #@param {type:"string"}
ALVO_DE_TREINO = "cipher387"  #@param {type:"string"}

PASTA_DIA = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
FERRAMENTA = PASTA_DIA / "br" / "03_mini_blackbird.py"
OFFLINE = os.environ.get("OSINT_OFFLINE") == "1"
SAIDAS = PASTA_DIA / "saidas"
SAIDAS.mkdir(exist_ok=True)

try:
    from zoneinfo import ZoneInfo
    FUSO = ZoneInfo("America/Sao_Paulo")
except Exception:
    FUSO = timezone(timedelta(hours=-3))


def variantes(apelido):
    """'kau.ofertas' → ['kau-ofertas', 'kau.ofertas', 'kau_ofertas', 'kauofertas']"""
    partes = apelido.replace("_", ".").replace("-", ".").split(".")
    return sorted({sep.join(partes) for sep in (".", "_", "-", "")})   # set: sem repetidos


def rodar(username):
    """Roda o mini-blackbird e devolve (código de retorno, linhas com resultado por site)."""
    comando = [sys.executable, str(FERRAMENTA), "--username", username]
    r = subprocess.run(comando, capture_output=True, text=True, encoding="utf-8", timeout=120,
                       cwd=PASTA_DIA, env=dict(os.environ, PYTHONIOENCODING="utf-8"))
    sites = [l for l in r.stdout.splitlines() if l.startswith(("GitHub", "GitLab", "Codeberg"))]
    return r.returncode, sites


# ── 1. Montar o lote ──────────────────────────────────────────────────────
lote = [(v, "simulação") for v in variantes(APELIDO_DO_CASO)] + [(ALVO_DE_TREINO, "execução")]
print(f"🧬 Variantes de {APELIDO_DO_CASO!r}: {', '.join(v for v, _ in lote[:-1])}")
print(f"🎯 Alvo de treino (público): {ALVO_DE_TREINO!r}\n")
if OFFLINE:
    print("⚠️ MODO OFFLINE: o alvo de treino será verificado com a amostra local.\n")

# ── 2. Processar ──────────────────────────────────────────────────────────
linhas = []
with open(SAIDAS / "log_lote.txt", "a", encoding="utf-8") as log:
    for username, modo in lote:
        quando = datetime.now(FUSO).isoformat(timespec="seconds")
        comando_txt = f"python br/03_mini_blackbird.py --username {username}"
        if modo == "simulação":
            print(f"📝 [simulação] {comando_txt}")
            linhas.append({"quando": quando, "username": username, "modo": modo,
                           "retorno": "", "resultado": "não consultado (apelido fictício)"})
        else:
            codigo, sites = rodar(username)
            print(f"▶️  [execução] {comando_txt}  → retorno {codigo}")
            for s in sites:
                print(f"     {s}")
            linhas.append({"quando": quando, "username": username, "modo": modo,
                           "retorno": codigo, "resultado": " | ".join(" ".join(s.split()) for s in sites)})
        log.write(f"{quando}\t{modo}\t{comando_txt}\n")

# ── 3. Salvar ─────────────────────────────────────────────────────────────
with open(SAIDAS / "lote_buscas.csv", "w", newline="", encoding="utf-8-sig") as f:
    escritor = csv.DictWriter(f, fieldnames=list(linhas[0]), delimiter=";")
    escritor.writeheader()
    escritor.writerows(linhas)

print(f"\n💾 saidas/lote_buscas.csv ({len(linhas)} linhas) e saidas/log_lote.txt gravados.")
print("⚖️  Num caso real, só rode o lote com autorização e finalidade definida, e trate cada")
print("   'encontrado' como hipótese: mesmo apelido não quer dizer mesma pessoa.")

## 🔎 Inovação OSINT do dia

**WhatsMyName: as regras de detecção como dados abertos.** O projeto comunitário
[WhatsMyName](https://github.com/WebBreacher/WhatsMyName) mantém um único arquivo, `wmn-data.json`, com
as regras de mais de 700 sites: a URL a consultar (`uri_check`), o status e o texto que indicam
"existe" (`e_code`, `e_string`) e os que indicam "não existe" (`m_code`, `m_string`). O blackbird usa
essa base, e o `br/03_mini_blackbird.py` copia a mesma ideia para três sites. Como é só JSON, dá para
ler com `urllib.request` + `json` (nativos) e montar o seu verificador. Licença CC BY-SA 4.0: cite a
fonte se reaproveitar.

## 📚 Para ir além

- Blackbird: <https://github.com/p1ngul1n0/blackbird> (documentação: <https://p1ngul1n0.gitbook.io/blackbird/>)
- WhatsMyName: <https://github.com/WebBreacher/WhatsMyName>
- Sherlock: <https://github.com/sherlock-project/sherlock> · Maigret: <https://github.com/soxoj/maigret>
- `argparse`, tutorial oficial: <https://docs.python.org/pt-br/3/howto/argparse.html>
- `subprocess`: <https://docs.python.org/pt-br/3/library/subprocess.html>

## ✅ Checklist

- [ ] Clonei o blackbird, instalei os requisitos e rodei `--username cipher387`
- [ ] Rodei o `launch.py` de dentro da pasta `blackbird/`
- [ ] Usei o `--help` da minha própria CLI de CEP
- [ ] Rodei o `02` duas vezes e vi o log crescer
- [ ] Entendi por que o GitLab precisa de checagem de texto
- [ ] Fiz o desafio S
- [ ] Anotei no meu [`DIARIO.md`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/DIARIO.md)

---

🎉 Terminou? [Registre sua entrega do Dia 3](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/issues/new?template=entreguei-o-dia.yml) · Baseado em [cipher387/python-for-OSINT-21-days](https://github.com/cipher387/python-for-OSINT-21-days) (MIT) · Adaptação pt-BR: [Ridd1kulusC0d3r/python-para-osint-21-dias](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias)